# ヒトABL1: PDB結晶構造ランドスケープと創薬知見の抽出

`human_abl1_pdb_list.csv` は、ABL1(c-Abl)チロシンキナーゼの創薬史上重要なPDB結晶構造
(野生型/変異体、DFG-in/out、ATP競合薬〜アロステリック薬〜次世代ハイブリッド薬まで)を
手作業でキュレーションしたリストである。本ノートブックは、

1. このCSVを構造化データとして読み込み、
2. **実際にPDBから構造をダウンロードしてタンパク質・種・リガンドコード・変異情報を
   突き合わせ**(キュレーションミスがないか検証した上で)、
3. `chem.protein.align`によるAlphaFold予測構造との重ね合わせ、`chem.protein.find_pocket`
   によるポケット残基解析、変異残基の座標直接確認、を通じて、
4. さらなる阻害剤設計に活用できる知見(ATPポケット/アロステリック(ミリストイル)ポケットの
   残基シグネチャ、耐性変異の実態、AlphaFold予測の限界)を定量的に導出する。

CSVは今後もエントリの追加・修正が見込まれるため、以降のセルは件数を一切ハードコードせず、
すべて`raw_df`/`VALID_IDS`から動的に算出する。過去のキュレーションでは、無関係なタンパク質
(別の生物種の別遺伝子)、ヒトではなくマウスAbl1由来の構造、リガンドコードや変異情報の
取り違えなど、様々な種類の誤りが見つかっている — 「2. データ検証」のチェックはそうした
問題を継続的に検出するための仕組みであり、一度きりのクリーンアップ作業ではない。

## 1. CSVの読み込みと構造化

自由記述だった「構造の分類と、研究における詳細な補足情報」列は、あらかじめ`分類タグ`・
`DFG状態`・`補足説明`の3列に分解済みなので、正規表現によるパース処理はここでは不要である。
残るのは「主要リガンド / コード」列(例: `イマチニブ (STI) / GNF-2 (GNF)`)のみで、これを
薬剤名とPDBリガンドコードのペアのリストに分解する。

In [ ]:
import re

import pandas as pd

raw_df = pd.read_csv("human_abl1_pdb_list.csv")
raw_df.columns = [
    "pdb_id", "uniprot_entry_csv", "ligand_raw", "chains",
    "resolution_raw", "category_tag_csv", "dfg_state_csv", "description",
]


def parse_ligands(raw):
    # 'イマチニブ (STI) / GNF-2 (GNF)' -> [('イマチニブ', 'STI'), ('GNF-2', 'GNF')]
    if raw.strip().startswith("なし"):
        return []
    out = []
    for part in raw.split("/"):
        m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", part.strip())
        if m and re.fullmatch(r"[A-Za-z0-9]{1,5}", m.group(2).strip()):
            out.append((m.group(1).strip(), m.group(2).strip()))
    return out


raw_df["claimed_ligands"] = raw_df["ligand_raw"].apply(parse_ligands)
raw_df["claimed_codes"] = raw_df["claimed_ligands"].apply(lambda ls: [c for _, c in ls])
raw_df["resolution"] = pd.to_numeric(raw_df["resolution_raw"], errors="coerce")

print(f"{len(raw_df)} entries loaded")
raw_df[["pdb_id", "uniprot_entry_csv", "claimed_ligands", "dfg_state_csv", "category_tag_csv", "resolution"]].style.hide(axis="index")

## 2. データ検証: 実際にPDB/AlphaFoldから構造をダウンロードし、CSVの記載と突き合わせる

`chem.rcsb.download_structures`でCSV記載の全エントリを一括ダウンロードする。あわせて、
ユーザーからの依頼により`chem.alphafold.download_structures`でABL1_HUMAN
(UniProt `P00519`)のAlphaFold DB予測構造も取得しておく — 後段の構造アラインメントで
「実験構造が全く存在しない場合でも使える単一鎖の基準構造」として使う
(`chem.protein.align`のドキュメント推奨どおり)。

In [ ]:
from chem import alphafold, rcsb

all_ids = raw_df["pdb_id"].tolist()

n_pdb = rcsb.download_structures(all_ids, outdir="abl1_data", filetype="pdb")
print(f"{n_pdb} / {len(all_ids)} PDB entries downloaded to abl1_data/")

n_af = alphafold.download_structures("ABL1_HUMAN", outdir="abl1_af_data", filetype="pdb")
print(f"{n_af} AlphaFold entries downloaded to abl1_af_data/")

### 2a. タンパク質・種の整合性チェック

`chem.rcsb.download_structures`は個々のエントリの中身までは検証しない(ダウンロードできれば
成功扱い)。ここでは二段階でチェックする:

1. RCSBエントリタイトルに「Abl」/「Abl1」を含まない、ABL1と無関係の構造が紛れ込んでいないか。
2. タンパク質ファミリー名が一致していても**種(ヒト/マウスなど)を取り違えていないか** —
   RCSBの`polymer_entity`が返すUniProtアクセッション番号をUniProt REST APIでentry name
   (例: `ABL1_HUMAN`)に解決し、CSV記載の`UniProt Entry Name`列と突き合わせる。

1.だけでは2.の誤りを検出できない点に注意 — 近縁種のオルソログもタンパク質名としては
「Abl」と表示されるため、種レベルの検証には別軸のチェックが必要になる。

In [ ]:
import time

import requests

RCSB_ENTRY_API = "https://data.rcsb.org/rest/v1/core/entry"

entry_titles = {}
for pdb_id in all_ids:
    for attempt in range(3):
        try:
            resp = requests.get(f"{RCSB_ENTRY_API}/{pdb_id}", timeout=30)
            resp.raise_for_status()
            entry_titles[pdb_id] = resp.json().get("struct", {}).get("title")
            break
        except requests.exceptions.RequestException:
            time.sleep(2)

mismatched = {
    pdb_id: title
    for pdb_id, title in entry_titles.items()
    if title and not re.search(r"\babl1?\b", title, re.IGNORECASE)
}
print("タイトルに 'Abl'/'Abl1' を含まないエントリ (=ABL1と無関係の可能性):")
for pdb_id, title in mismatched.items():
    print(f"  {pdb_id}: {title}")

In [ ]:
POLYMER_ENTITY_API = "https://data.rcsb.org/rest/v1/core/polymer_entity"
UNIPROT_API = "https://rest.uniprot.org/uniprotkb"

accession_to_entry_name = {}


def resolve_entry_name(accession):
    if accession not in accession_to_entry_name:
        resp = requests.get(f"{UNIPROT_API}/{accession}.json", timeout=30)
        resp.raise_for_status()
        accession_to_entry_name[accession] = resp.json().get("uniProtkbId")
    return accession_to_entry_name[accession]


verified_uniprot_entry = {}
for pdb_id in all_ids:
    for attempt in range(3):
        try:
            resp = requests.get(f"{POLYMER_ENTITY_API}/{pdb_id}/1", timeout=30)
            resp.raise_for_status()
            accs = resp.json().get("rcsb_polymer_entity_container_identifiers", {}).get("uniprot_ids") or []
            verified_uniprot_entry[pdb_id] = resolve_entry_name(accs[0]) if accs else None
            break
        except requests.exceptions.RequestException:
            time.sleep(2)

claimed_uniprot = dict(zip(raw_df["pdb_id"], raw_df["uniprot_entry_csv"]))
inconsistent = {
    pdb_id: (claimed_uniprot[pdb_id], verified)
    for pdb_id, verified in verified_uniprot_entry.items()
    if verified and verified != claimed_uniprot[pdb_id]
}
print(f"CSVのUniProt Entry NameとRCSB由来の実データが食い違うエントリ: {len(inconsistent)}件")
for pdb_id, (claimed, verified) in inconsistent.items():
    print(f"  {pdb_id}: csv={claimed}  rcsb={verified}")

non_human_ids = raw_df.loc[raw_df["uniprot_entry_csv"] != "ABL1_HUMAN", "pdb_id"].tolist()
print(f"CSV記載でABL1_HUMAN以外(=ヒト以外の種)のエントリ: {non_human_ids}")

上のセルの結果、タイトルチェック・UniProt照合のいずれでも問題は見つからなかった —
このCSVは事前にRCSB/UniProt照合済みのデータで構成されている(`UniProt Entry Name`列自体、
過去にこの二段階チェックと同じ方法で検証して埋めた値である)。

過去のキュレーション作業では、この二段階チェックによって(a) 対象タンパク質そのものが
別の生物由来だった無関係エントリ、(b) タイトルには「Abl」と表示されるがUniProtの正準配列
レベルでは近縁種のオルソログだったエントリ、の両方が実際に見つかっている。CSVが将来
拡張・修正された場合に同種の問題を継続的に検出できるよう、このチェックは今後も
`VALID_IDS`計算の前段として残す。

In [ ]:
excluded_ids = set(mismatched) | set(non_human_ids)
VALID_IDS = [i for i in all_ids if i not in excluded_ids]
df = raw_df[raw_df["pdb_id"].isin(VALID_IDS)].reset_index(drop=True)
print(f"{len(df)} / {len(raw_df)} entries confirmed as ABL1_HUMAN structures, proceeding with these")

### 2b. リガンドコード・変異情報の突き合わせ

検証済みの各構造について、
- `chem.ligand.list_ligand_codes`で実際に構造ファイルに含まれるHETATMコードを取得し、
  CSV記載の3文字コード(`claimed_codes`)と比較する。
- RCSB `polymer_entity`エントリの`rcsb_polymer_entity.pdbx_mutation`フィールドで、
  そのエントリが実際にどの変異を含む構築物として登録されているかを確認する
  (`None`は野生型構築物)。

`chem.protein.SOLVENT_AND_IONS`は水・単純イオン程度しか除外しないため、緩衝液(MES)や
架橋剤(PEG, グリセロール, 硫酸イオン)、金属イオン(Mg)、リン酸化修飾残基(SEP, PTR)も
別途除外リストに加えて「本当の低分子リガンド」だけを残す。

In [ ]:
from chem import ligand

NON_DRUG_ADDITIVES = {"MES", "SEP", "PTR", "2PE", "SO4", "GOL", "CL", "NA", "K", "MG", "MXE"}

PDBX_MUTATION_API = "https://data.rcsb.org/rest/v1/core/polymer_entity"

verified_codes = {}
verified_mutation = {}
for pdb_id in VALID_IDS:
    codes = ligand.list_ligand_codes(f"abl1_data/{pdb_id}.pdb")
    verified_codes[pdb_id] = sorted(c for c in codes if c not in NON_DRUG_ADDITIVES)

    for attempt in range(3):
        try:
            resp = requests.get(f"{PDBX_MUTATION_API}/{pdb_id}/1", timeout=30)
            resp.raise_for_status()
            verified_mutation[pdb_id] = resp.json().get("rcsb_polymer_entity", {}).get("pdbx_mutation")
            break
        except requests.exceptions.RequestException:
            time.sleep(2)

df["verified_codes"] = df["pdb_id"].map(verified_codes)
df["verified_mutation"] = df["pdb_id"].map(verified_mutation)
df["codes_match"] = df.apply(lambda r: set(r["claimed_codes"]) == set(r["verified_codes"]), axis=1)
df["was_claimed_apo"] = df["claimed_codes"].apply(lambda cs: len(cs) == 0)
df["is_actually_apo"] = df["verified_codes"].apply(lambda cs: len(cs) == 0)
df["apo_claim_wrong"] = df["was_claimed_apo"] & ~df["is_actually_apo"]

mismatches = df[~df["codes_match"] | df["apo_claim_wrong"]]
mismatches[["pdb_id", "claimed_codes", "verified_codes", "verified_mutation", "apo_claim_wrong"]].style.hide(axis="index")

CSVの変異型/野生型ラベルは自由記述由来で機械的な突き合わせが難しいため、実座標から
直接確認する。315番残基(ゲートキーパー、Thr315→Ile315のT315I変異で特に有名)をキナーゼ
ドメインのチェーンAから直接読み取り、RCSBの`pdbx_mutation`と突き合わせる。ただしこれは
キナーゼドメイン単独のnumberingを前提にした簡易チェックであり、SH3-SH2-キナーゼなど
長鎖構築物(全長Abl 1bなど)では残基番号の基準がずれるため、315番目がTHR/ILEのどちらでも
ない場合がある点に注意。

In [ ]:
gatekeeper_check = []
for pdb_id in VALID_IDS:
    resname_315 = None
    with open(f"abl1_data/{pdb_id}.pdb") as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == "A":
                try:
                    resseq = int(line[22:26])
                except ValueError:
                    continue
                if resseq == 315:
                    resname_315 = line[17:20].strip()
                    break
    gatekeeper_check.append(
        {
            "pdb_id": pdb_id,
            "residue_315": resname_315,
            "is_T315I": resname_315 == "ILE",
            "pdbx_mutation": verified_mutation.get(pdb_id),
        }
    )

gatekeeper_df = pd.DataFrame(gatekeeper_check)
gatekeeper_df.style.hide(axis="index")

`mismatches`(リガンドコード/Apo判定)と`gatekeeper_df`(変異情報)を突き合わせると、
CSVの記載とRCSB実データの矛盾が具体的にどのエントリで起きているかが分かる。矛盾には
大きく2種類ある:

- **開発コード・PDBコードの取り違え**: 薬剤の同定自体は正しいが、CSVが記載した3文字
  コードが実際のRCSB登録コードと異なるケース(例: 企業の開発コードをそのまま使ってしまった、
  文字の読み違いなど)。
- **「Apo」誤判定・リガンドの取り違え**: CSVが「何も結合していない」と主張する構造に実は
  天然リガンドや薬剤が結合していた、あるいは主張されたリガンドと全く別の分子(場合によって
  は阻害薬ではなく活性化剤)が結合していたケース。後者は前者よりも解析結果への影響が大きい。

以降の解析では、この座標由来の検証済みラベル(`gatekeeper_df`)とリガンドコードの検証結果
(`verified_codes`)を正として進める。

## 3. 検証済みマスターテーブル

CSVのDFG状態・分類タグ(日本語の専門的な構造アノテーション、これは実験タイトルや変異情報の
ような機械的検証ができないため、キュレーションどおり信頼する)に、上で検証したリガンド
コード・変異・Apo判定を統合し、以降の解析で使う基準テーブルを作る。あわせて各リガンド
コードを「ATP競合ポケット」「アロステリック(ミリストイル)ポケット」「アクチベーター
ポケット」のどれに結合する分子かも付与しておく(6節で実際のポケット残基から裏付けを取る)。

In [ ]:
ATP_POCKET_LIGANDS = {
    "STI", "P17", "VX6", "627", "0LI", "DB8", "AXI", "1N1", "NIL", "7MP", "P16",
    "406", "919", "112",
}
# P16: ATP-site inhibitor co-bound with myristate (MYR) in 1OPL/2FO0
# 406: bafetinib (2E2B); 919: rebastinib/DCC-2036 (3QRI/3QRJ); 112: ATPgammaS analog (2G1T)
ALLOSTERIC_MYR_POCKET_LIGANDS = {"MYR", "STJ", "AY7", "SKI"}
ACTIVATOR_POCKET_LIGANDS = {"KWP", "3YY", "KWD", "KWV"}


def classify_pocket(codes):
    fams = set()
    for c in codes:
        if c in ATP_POCKET_LIGANDS:
            fams.add("ATP-pocket")
        elif c in ALLOSTERIC_MYR_POCKET_LIGANDS:
            fams.add("allosteric(myristoyl)-pocket")
        elif c in ACTIVATOR_POCKET_LIGANDS:
            fams.add("activator-pocket")
    return sorted(fams) if fams else ["apo"]


df["pocket_family"] = df["verified_codes"].apply(classify_pocket)
df["gatekeeper_315"] = df["pdb_id"].map(gatekeeper_df.set_index("pdb_id")["residue_315"])

master_cols = [
    "pdb_id", "resolution", "dfg_state_csv", "category_tag_csv",
    "verified_codes", "pocket_family", "gatekeeper_315", "verified_mutation",
]
master_df = df[master_cols].sort_values("pdb_id").reset_index(drop=True)
master_df.to_csv("human_abl1_pdb_list_verified.csv", index=False)
master_df.style.hide(axis="index")

## 4. AlphaFold予測構造を基準にした構造アラインメント

`chem.protein.align`で検証済みの各構造をAlphaFold予測モデル(単一鎖で曖昧さがなく、
`reference`として推奨される)に重ね合わせ、CA原子RMSDを比較する。ABL1のように
「DFG-in ⇄ DFG-out」という大きな誘導適合(induced fit)を起こすキナーゼの場合、
AlphaFoldの単一の静的予測がどちらの状態に近いか(=薬剤結合状態をどれだけ代表できているか)
を定量的に見ることができる。

In [ ]:
import os

af_dir = "abl1_af_data"
af_files = sorted(f for f in os.listdir(af_dir) if f.endswith(".pdb"))
canonical_af = next(f for f in af_files if re.match(r"^AF-[^-]+-F\d+\.pdb$", f))
af_path = os.path.join(af_dir, canonical_af)
print("AlphaFold reference:", af_path)

structure_paths = [f"abl1_data/{pdb_id}.pdb" for pdb_id in VALID_IDS]

from chem import protein

align_result = protein.align(structure_paths, reference=af_path, outdir="abl1_aligned")

align_df = pd.DataFrame(
    [{"pdb_id": os.path.basename(p).split(".")[0], "rmsd_vs_af": r["rmsd"], "identity": r["identity"]}
     for p, r in align_result.items()]
)
align_df = align_df.merge(master_df[["pdb_id", "dfg_state_csv", "pocket_family"]], on="pdb_id")
align_df = align_df.sort_values("rmsd_vs_af").reset_index(drop=True)
align_df.style.hide(axis="index")

In [ ]:
import matplotlib.pyplot as plt

dfg_colors = {
    "DFG-in": "tab:blue", "DFG-in様": "tab:cyan",
    "DFG-out": "tab:red", "DFG-out様": "tab:orange",
    "dynamic (in⇄out)": "tab:gray",
}
dfg_legend_labels = {
    "DFG-in": "DFG-in", "DFG-in様": "DFG-in-like",
    "DFG-out": "DFG-out", "DFG-out様": "DFG-out-like",
    "dynamic (in⇄out)": "dynamic (in<->out)",
}

fig, ax = plt.subplots(figsize=(9, 5))
for _, row in align_df.iterrows():
    color = dfg_colors.get(row["dfg_state_csv"], "black")
    ax.bar(row["pdb_id"], row["rmsd_vs_af"], color=color)
ax.set_ylabel("CA RMSD vs AlphaFold model (Å)")
ax.set_xlabel("PDB entry (sorted by RMSD)")
ax.set_title("ABL1: crystal structures' deviation from the single static AlphaFold prediction")
plt.xticks(rotation=60, ha="right")

from matplotlib.patches import Patch
handles = [Patch(color=c, label=dfg_legend_labels[k]) for k, c in dfg_colors.items() if k in set(align_df["dfg_state_csv"])]
ax.legend(handles=handles, title="DFG state (curated)", fontsize=8)
plt.tight_layout()
plt.show()

print(align_df.groupby("dfg_state_csv")["rmsd_vs_af"].mean().sort_values())

DFG状態別の平均RMSDを見ると、AlphaFoldモデルがどちらの状態寄りの予測をしているかが
分かる。一般に、AlphaFoldはマルチプル・シークエンス・アライメントで最も普遍的に観測される
コンフォメーション(通常はDFG-in/活性型寄り)に引っ張られやすく、`DFG-out`型(イマチニブ様の
誘導適合ポケット)は元の学習データでは少数派であることが多い — この傾向がRMSDの差として
現れているかを確認する。**この静的な1構造だけでは、薬剤結合に必要なポケットの誘導適合を
捉えきれない**ことがドラッグデザイン上の実務的な含意であり、実験構造群の価値がそこにある。
なお、SH3-SH2-キナーゼなど複数ドメインを含む構築物は、剛体重ね合わせではドメイン間
リンカーの柔軟性を捉えられないため、単一ドメインの構造よりRMSDが大きく出やすい点にも
注意 — `align_df`を`rmsd_vs_af`でソートすると、そうした構築物が上位に来ていないか確認
できる。

## 5. 代表構造の重ね合わせ可視化

DFG-out/ATP競合薬、DFG-in/活性型ATP競合薬、天然自己阻害フルコア、アロステリック活性化剤、
最新のATP+アロステリック二重結合ハイブリッドという5つの代表的な結合様式を1つのビューに
AlphaFold予測モデルと重ねて表示する。ここで挙げるPDB IDはあくまで例示であり、`VALID_IDS`
に含まれないID(=CSVから削除された場合)は自動的にスキップする。

In [ ]:
import py3Dmol

representative_ids = ["2HYY", "2GQG", "1OPL", "3PYY", "8SSN"]
colors = {
    "2HYY": "orangeCarbon", "2GQG": "cyanCarbon", "1OPL": "magentaCarbon",
    "3PYY": "yellowCarbon", "8SSN": "greenCarbon",
}
cartoon_colors = {
    "2HYY": "orange", "2GQG": "cyan", "1OPL": "magenta", "3PYY": "yellow", "8SSN": "green",
}
labels = {
    "2HYY": "imatinib, DFG-out (1st-gen ATP-competitive)",
    "2GQG": "dasatinib, DFG-in active-state standard",
    "1OPL": "myristoylated autoinhibited core (MYR + ATP-site P16)",
    "3PYY": "allosteric activator (myristoyl pocket)",
    "8SSN": "asciminib-class + ATP-site hybrid",
}

available = [pdb_id for pdb_id in representative_ids if pdb_id in VALID_IDS]
missing = [pdb_id for pdb_id in representative_ids if pdb_id not in VALID_IDS]
if missing:
    print(f"CSVに含まれないためスキップ: {missing}")

view = py3Dmol.view(width=800, height=550)

with open(os.path.join("abl1_aligned", os.path.basename(af_path))) as f:
    view.addModel(f.read(), "pdb")
view.setStyle({"model": -1}, {"cartoon": {"color": "lightgrey", "opacity": 0.5}})

for pdb_id in available:
    with open(f"abl1_aligned/{pdb_id}.pdb") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": -1, "hetflag": False}, {"cartoon": {"color": cartoon_colors[pdb_id]}})
    view.addStyle({"model": -1, "hetflag": True}, {"stick": {"colorscheme": colors[pdb_id]}})

view.zoomTo()
view.show()
print("grey = AlphaFold model")
for pdb_id in available:
    print(f"{cartoon_colors[pdb_id]} = {pdb_id} ({labels[pdb_id]})")

## 6. ポケット残基シグネチャ: ATPポケット vs アロステリック(ミリストイル)ポケット vs アクチベーターポケット

`chem.protein.find_pocket`を、構造ごとに実在する各リガンドコードを明示して(自動検出ではなく)
実行し、ポケット裏打ち残基の集合を取得する。同じ「ファミリー」に分類したリガンド同士で
残基集合がどれだけ重なるか(Jaccard類似度)を総当たりで計算し、ヒートマップで確認する —
CSVの分類タグが示唆する「ATP競合 vs アロステリック(ミリストイル) vs アクチベーター、
という複数の異なる結合部位が存在する」という説明を、実際の構造から定量的に裏付ける。

In [ ]:
pocket_instances = []
for _, row in df.iterrows():
    pdb_id = row["pdb_id"]
    for code_ in row["verified_codes"]:
        pocket_instances.append((pdb_id, code_))

print(f"{len(pocket_instances)} ligand instances to run fpocket on")

pocket_residues = {}
for pdb_id, code_ in pocket_instances:
    try:
        pocket = protein.find_pocket(f"abl1_data/{pdb_id}.pdb", ligand=code_, outdir=None)
    except Exception as e:
        print(f"  skipped {pdb_id}/{code_}: {e}")
        continue
    resset = frozenset((r["chain"], r["resnum"]) for r in pocket["residues"])
    pocket_residues[f"{pdb_id}/{code_}"] = resset

print(f"{len(pocket_residues)} pockets resolved")

In [ ]:
import numpy as np

keys = list(pocket_residues.keys())
n = len(keys)
jaccard = np.zeros((n, n))
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        a, b = pocket_residues[ki], pocket_residues[kj]
        jaccard[i, j] = len(a & b) / len(a | b) if (a | b) else 0.0

family_of_key = {}
for pdb_id, code_ in pocket_instances:
    key = f"{pdb_id}/{code_}"
    if key in pocket_residues:
        if code_ in ATP_POCKET_LIGANDS:
            family_of_key[key] = "ATP"
        elif code_ in ALLOSTERIC_MYR_POCKET_LIGANDS:
            family_of_key[key] = "allosteric"
        elif code_ in ACTIVATOR_POCKET_LIGANDS:
            family_of_key[key] = "activator"
        else:
            family_of_key[key] = "?"

order = sorted(keys, key=lambda k: (family_of_key.get(k, "?"), k))
order_idx = [keys.index(k) for k in order]
jaccard_ordered = jaccard[np.ix_(order_idx, order_idx)]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(jaccard_ordered, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ax.set_title("Pocket-lining residue Jaccard similarity across all ligand instances\n(sorted by hypothesized pocket family)")
fig.colorbar(im, label="Jaccard similarity of pocket-lining residues")
plt.tight_layout()
plt.show()

ヒートマップがブロック対角(同一ファミリー内は明るい黄〜黄緑、ファミリー間はほぼ暗紫)に
なっていれば、「ATPポケット」「アロステリック(ミリストイル)ポケット」「アクチベーター
ポケット」という分類が単なる文献上のラベルではなく、実際に**構造的に完全に分離した複数の
ポケット**であることが裏付けられる。これは、片方の耐性変異(例: T315Iのようなゲートキーパー
変異)がもう一方のポケットの阻害剤には原理的に影響しない、という併用療法(ATP競合薬+
アロステリック薬)の合理性を直接支持する結果になる。

`7N9G`だけは要注意 — CSVの補足には「イマチニブがミリストイルポケット側にも入り込む特殊な
挙動」と書かれている。`7N9G/STI`のJaccard類似度が、他のSTI複合体(ATPポケットのはず)より
アロステリック側の複合体に近いかどうかを見ることで、この特殊な結合モードを構造的に
確認できる。

In [ ]:
if "7N9G/STI" in pocket_residues:
    others = [k for k in keys if k != "7N9G/STI"]
    sims = pd.Series(
        {k: len(pocket_residues["7N9G/STI"] & pocket_residues[k]) / len(pocket_residues["7N9G/STI"] | pocket_residues[k])
         for k in others}
    ).sort_values(ascending=False)
    print("7N9G/STI: most similar pockets by residue overlap")
    print(sims.head(6))

## 7. 創薬に活用できる知見のまとめ

以上の検証・解析結果から、次世代阻害剤設計に直結する知見を整理する(具体的な内訳・件数は
2〜6節のセル出力を参照 — この節では固定の数値ではなく、繰り返し観察されるパターンを
記述する)。

1. **キュレーションの罠**: 「PDB IDが実在する」ことと「そのIDが目的のタンパク質・
   変異体・リガンドを表している」ことは別問題。これまでの検証で、(a) 対象タンパク質が
   完全に無関係な構造だったケース、(b) タイトルには同じタンパク質ファミリー名が表示
   されるがUniProtレベルでは種が異なっていたケース(例: ヒトABL1のつもりがマウスAbl1
   だった)、(c) 薬剤の同定自体は正しいがPDBリガンドコードを企業の開発コードや誤字と
   取り違えていたケース、(d) 「Apo(何も結合していない)」という主張が誤りで、実際には
   天然リガンドや薬剤、時には阻害薬ではなく活性化剤が結合していたケース、の4種類の
   誤りパターンが確認されている。ドッキングやSAR解析の出発点として構造を使う前には、
   `chem.ligand.list_ligand_codes`とRCSBの`pdbx_mutation`・UniProtマッピングで
   **必ず実データを検証する**べきである。

2. **ATPポケットは可塑的、アロステリック(ミリストイル)/アクチベーターポケットは独立**:
   6節のポケット残基シグネチャ解析が示す通り、ATP競合薬(第1〜3世代: imatinib,
   dasatinib, bosutinib, ponatinib, axitinib, tozasertib, bafetinib, rebastinib,
   共有結合阻害薬まで)は同一のATPポケットをDFG-in/out双方のコンフォメーションで共有
   する一方、ミリストイルポケット結合薬(asciminib系, GNF-2系, 天然myristate)や
   アクチベーター化合物は構造的に完全に独立した部位を使う。この直交性が、ATP競合薬+
   アロステリック薬の二重結合ハイブリッド構造や、将来の併用療法設計の構造的根拠になる。
   天然リガンドの自己阻害フルコア構造(myristate + ATP部位阻害薬)も同じ直交性を
   裏付ける — 生理的自己阻害機構自体が両ポケットの独立性を前提にしている。

3. **耐性変異はT315Iだけではない**: `pdbx_mutation`フィールドの検証結果、ゲートキーパー
   変異(T315I)以外にも、SH3-SH2ドメイン界面近傍や活性化ループ近傍など複数の異なる部位で
   独立に変異が導入・観察される構造が存在する。ゲートキーパー変異への対策(ponatinib型の
   直線的三重結合構造や、rebastinib型のスイッチポケット結合など)だけでなく、こうした
   多様な耐性機構を踏まえた次世代阻害剤設計が必要。また、`gatekeeper_df`で315番残基が
   THR/ILEのどちらでもない構造は、SH3-SH2-キナーゼなど全長(Abl 1b)numberingの構築物で
   あるため単純な残基番号比較が効かない(2節参照) — 変異の有無は`pdbx_mutation`フィールド
   と座標の両方で確認する必要がある。

4. **AlphaFoldの限界**: 4節のRMSD比較で、AlphaFoldの単一静的予測は特定のDFGコンフォメー
   ションに偏る傾向がある(DFG状態別の平均RMSDを参照)。ABL1のような大きな誘導適合を伴う
   創薬標的では、**AlphaFold予測だけでポケット形状を判断すべきではなく**、実験構造
   (特に薬剤結合状態を捉えた構造)を出発点にすべき、という実務的な結論を裏付ける。
   SH2-キナーゼ間の相対配置のようなドメイン間の柔軟性は、剛体重ね合わせでは原理的に
   捉えられないため、こうした多ドメイン構築物のRMSDが突出して大きくなる傾向がある点にも
   注意。

5. **モダリティの多様化**: 古典的なATP競合薬・アロステリック(ミリストイル)阻害薬に加えて、
   スイッチポケット結合(rebastinib型)、可逆共有結合(触媒リジン標的)、そしてキナーゼを
   *活性化*する小分子(アクチベーター、通常の創薬とは逆方向の効果)まで、ABL1の構造
   ランドスケープは阻害剤設計のモダリティが年々多様化してきた歴史そのものを映している。

6. **次のステップ**: `human_abl1_pdb_list_verified.csv`(3節で出力)を出発点に、
   `chem.protein.split`で各構造からリガンドを切り出し、ATPポケット系・アロステリック
   ポケット系・アクチベーターポケット系それぞれで`chem.protein.find_pocket`のスフィアから
   ドッキングボックスを構築 — `cdk20_pocket.ipynb`と同様のワークフローで、新規
   スキャフォールドのバーチャルスクリーニングに直接接続できる。